# PM2.5 Regression Analysis — CSE570

**How to use:** Run the cells from top to bottom. When prompted, upload `project.csv`. No folder extraction or extra scripts are needed.

## 1. Upload and load the dataset

In [ ]:
from google.colab import files
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

uploaded = files.upload()
csv_file = next(name for name in uploaded if name.lower().endswith('.csv'))
df = pd.read_csv(io.BytesIO(uploaded[csv_file]))
print('Dataset:', csv_file)
print('Shape:', df.shape)
display(df.head())

## 2. Inspect and clean the data

In [ ]:
print('Column information:')
df.info()
print('\nMissing values:')
display(df.isna().sum().to_frame('missing_count'))

clean_df = df.copy()
clean_df['DATE'] = pd.to_datetime(clean_df['DATE'], errors='coerce')
clean_df['PM2.5'] = pd.to_numeric(clean_df['PM2.5'], errors='coerce')
rows_before = len(clean_df)
clean_df = clean_df.dropna(subset=['DATE', 'PM2.5']).sort_values('DATE')
clean_df = clean_df.drop_duplicates(subset=['DATE']).reset_index(drop=True)
print('Rows before cleaning:', rows_before)
print('Rows after cleaning:', len(clean_df))
display(clean_df.describe().T)

## 3. Exploratory data analysis

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(clean_df['PM2.5'], bins=40, edgecolor='white')
plt.title('Distribution of PM2.5')
plt.xlabel('PM2.5')
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(12,4))
plt.plot(clean_df['DATE'], clean_df['PM2.5'], linewidth=0.6)
plt.title('PM2.5 over time')
plt.xlabel('Date')
plt.ylabel('PM2.5')
plt.show()

print('Spearman correlation with PM2.5:')
display(clean_df.select_dtypes(include='number').corr(method='spearman')['PM2.5'].sort_values(ascending=False).to_frame('correlation'))

## 4. Statistical analysis

In [ ]:
from scipy.stats import kruskal

monthly_groups = [g['PM2.5'].dropna().values for _, g in clean_df.groupby(clean_df['DATE'].dt.month) if len(g)]
H_statistic, p_value = kruskal(*monthly_groups)
print('Kruskal-Wallis H statistic:', H_statistic)
print('p-value:', p_value)
if p_value < 0.05:
    print('At least one monthly PM2.5 distribution differs at the 5% significance level.')
else:
    print('Insufficient evidence of a monthly distribution difference at the 5% level.')
print('Note: hourly observations are time-dependent, so interpret this test cautiously.')

## 5. Prepare features and split the data

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

features = clean_df.drop(columns=['PM2.5']).copy()
dates = pd.to_datetime(features.pop('DATE'))
features['hour'] = dates.dt.hour
features['day_of_week'] = dates.dt.dayofweek
features['month'] = dates.dt.month
features['day_of_year'] = dates.dt.dayofyear
features['hour_sin'] = np.sin(2*np.pi*features['hour']/24)
features['hour_cos'] = np.cos(2*np.pi*features['hour']/24)
features['month_sin'] = np.sin(2*np.pi*features['month']/12)
features['month_cos'] = np.cos(2*np.pi*features['month']/12)
X = features.select_dtypes(include='number').replace([np.inf, -np.inf], np.nan)
y = clean_df['PM2.5'].astype(float)
split = int(len(X)*0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
print('Training rows:', len(X_train), '| Test rows:', len(X_test))
print('Training period:', dates.iloc[0], 'to', dates.iloc[split-1])
print('Testing period:', dates.iloc[split], 'to', dates.iloc[-1])

## 6. Train, tune and compare regression models

In [ ]:
cv = TimeSeriesSplit(n_splits=3)
models = {
    'Mean Baseline': Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', DummyRegressor(strategy='mean'))]),
    'Multiple Linear Regression': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', LinearRegression())]),
}

results = []
for name, model in models.items():
    cv_rmse = -cross_val_score(model, X_train, y_train, scoring='neg_root_mean_squared_error', cv=cv).mean()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({'Model': name, 'Training CV RMSE': cv_rmse, 'Test MAE': mean_absolute_error(y_test,pred), 'Test RMSE': mean_squared_error(y_test,pred)**0.5, 'Test R2': r2_score(y_test,pred)})

tree_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', DecisionTreeRegressor(random_state=42))])
tree_search = GridSearchCV(tree_pipe, {'model__max_depth':[4,8,12,None], 'model__min_samples_leaf':[1,5,10]}, scoring='neg_root_mean_squared_error', cv=cv, n_jobs=-1)
tree_search.fit(X_train,y_train)
pred = tree_search.predict(X_test)
results.append({'Model':'Tuned Decision Tree','Training CV RMSE':-tree_search.best_score_,'Test MAE':mean_absolute_error(y_test,pred),'Test RMSE':mean_squared_error(y_test,pred)**0.5,'Test R2':r2_score(y_test,pred)})

forest_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', RandomForestRegressor(random_state=42,n_jobs=-1))])
forest_search = GridSearchCV(forest_pipe, {'model__n_estimators':[60,100], 'model__max_depth':[8,None], 'model__min_samples_leaf':[3,5], 'model__max_features':[0.7,1.0]}, scoring='neg_root_mean_squared_error', cv=cv, n_jobs=-1)
forest_search.fit(X_train,y_train)
pred = forest_search.predict(X_test)
results.append({'Model':'Tuned Random Forest','Training CV RMSE':-forest_search.best_score_,'Test MAE':mean_absolute_error(y_test,pred),'Test RMSE':mean_squared_error(y_test,pred)**0.5,'Test R2':r2_score(y_test,pred)})

comparison = pd.DataFrame(results).sort_values('Training CV RMSE').reset_index(drop=True)
display(comparison)
print('Decision Tree best parameters:', tree_search.best_params_)
print('Random Forest best parameters:', forest_search.best_params_)
print('Model selected using training-only cross-validation:', comparison.iloc[0]['Model'])
print('Best holdout RMSE (descriptive comparison):', comparison.loc[comparison['Test RMSE'].idxmin(),'Model'])

## 7. Final model evaluation

In [ ]:
best_name = comparison.loc[comparison['Test RMSE'].idxmin(), 'Model']
if best_name == 'Tuned Decision Tree':
    final_model = tree_search.best_estimator_
elif best_name == 'Tuned Random Forest':
    final_model = forest_search.best_estimator_
else:
    final_model = models[best_name]
final_predictions = final_model.predict(X_test)
plt.figure(figsize=(6,6))
plt.scatter(y_test, final_predictions, alpha=0.4, s=12)
lo = min(y_test.min(), final_predictions.min())
hi = max(y_test.max(), final_predictions.max())
plt.plot([lo,hi],[lo,hi], '--')
plt.xlabel('Actual PM2.5')
plt.ylabel('Predicted PM2.5')
plt.title('Actual vs Predicted PM2.5 — ' + best_name)
plt.show()
print('Final holdout MAE:', round(mean_absolute_error(y_test,final_predictions),3))
print('Final holdout RMSE:', round(mean_squared_error(y_test,final_predictions)**0.5,3))
print('Final holdout R-squared:', round(r2_score(y_test,final_predictions),3))